In [ ]:
import numpy as np
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import json

from lac.perception.segmentation import SemanticClasses, UnetSegmentation
from lac.perception.segmentation_util import LAC_LABEL_COLORS, color_to_label, label_to_color
from lac.utils.plotting import plot_heatmap
from lac.util import load_data, load_stereo_images, load_images
from lac.utils.visualization import image_grid
from lac.params import LAC_DATA_PATH

%load_ext autoreload
%autoreload 2

## Load data


In [ ]:
run_name = "semantics_map1_preset2_recovery_agent"
data_path = LAC_DATA_PATH / "segmentation" / run_name
initial_pose, lander_pose, poses, imu_data, cam_config, json_data = load_data(data_path)
config = json.load(open("../../configs/nine_loops.json"))
print(f"Loaded {len(poses)} poses")

In [ ]:
# left_imgs, right_imgs = load_stereo_images(data_path, start_frame=0, end_frame=2000)
# images = {"FrontLeft": left_imgs, "FrontRight": right_imgs}

images = load_images(
    data_path,
    cameras=["FrontLeft", "FrontRight", "FrontLeft_semantic", "FrontRight_semantic"],
    start_frame=0,
    end_frame=1000,
)

## Run segmentation


In [ ]:
segmentation = UnetSegmentation()

In [ ]:
FRAME = 100

img = images["FrontLeft"][FRAME]
pred_label = segmentation.predict(img)
pred_color = label_to_color(pred_label)
gt_color = images["FrontLeft_semantic"][FRAME]
gt_label = color_to_label(gt_color)

In [ ]:
pred_vis = label_to_color(pred_label, custom=True)
gt_vis = label_to_color(gt_label, custom=True)

In [ ]:
error = pred_label != gt_label
error_vis = 255 * np.ones((*error.shape, 3), dtype=np.uint8)
error_vis[error] = (255, 0, 0)  # Red where there are errors

In [ ]:
image_grid([img, gt_vis, pred_vis, error_vis], rows=1, cols=4, figsize=(60, 10))